# MHRAG: run every GPU experiment on Kaggle

Kaggle gives about 30 free GPU hours per week (T4 ×2 or P100, 12-hour sessions), and it's the most practical free option for
the full multi-LLM benchmark.

**One-time notebook settings (right-hand panel):**
1. **Accelerator:** *GPU T4 x2* (or *GPU P100*).
2. **Internet:** *On*. You need to verify your phone number with Kaggle once to enable it.
3. **Secrets** (*Add-ons → Secrets*, optional): `HF_TOKEN` (needed for the gated Llama/Gemma models; accept their licences on
   Hugging Face first) and `GROQ_API_KEY` (API reference models and the LLM judge). Tick each secret for this notebook.

**Resuming across sessions.** Everything is written to `/kaggle/working/mhrag_results`. When a session ends, *Save Version →
Save & Run All (Commit)* or download the output. To resume, add that notebook's output as an **input dataset**. Cell 4 copies
previous results back, and every experiment skips answers it already has.

Every number the scripts write is a real measurement. Nothing is filled in by hand.

**Moved from the laptop run:** the chunk-size retrieval ablation (cell 9) and the local-model generation experiment (cells 10b). Retrieval main is already computed and committed.


In [ ]:
# 1) GPU and internet check
!nvidia-smi || echo "No GPU: set Accelerator to GPU T4 x2 in the notebook settings"
!curl -sI https://huggingface.co | head -1 || echo "No internet: turn Internet on in the notebook settings"

In [ ]:
# 2) Clone the repository
REPO_URL = "https://github.com/HellDragger/MentalHealthRAG-Chatbot.git"   # change if you forked it
BRANCH = "v2-research"
!rm -rf /kaggle/working/mhrag && git clone -b $BRANCH --depth 1 $REPO_URL /kaggle/working/mhrag
%cd /kaggle/working/mhrag

In [ ]:
# 3) Install (GPU extras + evaluation tools + 4-bit loading). Kaggle images already ship torch with CUDA.
!pip -q install -e ".[gpu,eval,dev]" bitsandbytes

In [ ]:
# 4) Secrets, results directory, and resume from a previous run's output (attached as an input dataset)
import os, glob, shutil
try:
    from kaggle_secrets import UserSecretsClient
    sc = UserSecretsClient()
    for k in ("HF_TOKEN", "GROQ_API_KEY", "OPENROUTER_API_KEY"):
        try:
            os.environ[k] = sc.get_secret(k)
        except Exception:
            pass
except ImportError:
    print("not running on Kaggle")

RESULTS = "/kaggle/working/mhrag_results"
os.makedirs(RESULTS, exist_ok=True)
prev = glob.glob("/kaggle/input/*/mhrag_results")
if prev:
    shutil.copytree(prev[0], RESULTS, dirs_exist_ok=True)
    print("resumed from", prev[0])
os.environ["MHRAG_PATHS__RESULTS"] = RESULTS
os.environ["MHRAG_LOAD_IN_4BIT"] = "1"          # NF4 for 7-12B models on a 16 GB T4
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print({k: bool(os.environ.get(k)) for k in ("HF_TOKEN", "GROQ_API_KEY")})

In [ ]:
# 5) Offline test suite (sanity check)
!pytest -q

In [ ]:
# 6) Indexes: the default one plus the full embedder x chunk-size grid (idempotent)
!python -m scripts.build_index
!python -m scripts.build_index --grid

In [ ]:
# 7) Evaluation sets. The sets used for the committed results (incl. SynthQA, 189 questions generated with
#    Qwen2.5-1.5B) are in the repo under eval/data/, so all experiments use identical queries. Set
#    REGENERATE_SYNTH = True only for a new, stronger SynthQA (then re-run every retrieval experiment).
REGENERATE_SYNTH = False
if REGENERATE_SYNTH:
    import os
    gen = "llama-3.3-70b-groq" if os.environ.get("GROQ_API_KEY") else "qwen2.5-7b-instruct"
    !python -m scripts.make_eval_sets --synth --synth-model $gen --synth-n 300
!ls -la eval/data/


In [ ]:
# 8) Risk classifiers (DistilRoBERTa fine-tunes in a few minutes on a T4) + safety-gate evaluation
!python -m scripts.train_risk_classifier
# With access to the gated MentalRoBERTa (request it on its HF page first):
# !python -m scripts.train_risk_classifier --transformer-model mental/mental-roberta-base
!python -m scripts.eval_safety --tag v2_devset
!python -m scripts.eval_safety --data eval/data/safety_prompts_heldout.jsonl --tag v2_heldout

In [ ]:
# 9) Retrieval experiments
# retrieval_main was already run on the laptop and its results are committed (results/retrieval_main.json);
# set RUN_RETRIEVAL_MAIN = True to recompute it here. The chunk-size ablation (128/256/512 tokens x 5 embedders)
# was moved to Kaggle.
RUN_RETRIEVAL_MAIN = False
if RUN_RETRIEVAL_MAIN:
    !python -m scripts.run_eval --config configs/experiments/retrieval_main.yaml
!python -m scripts.run_eval --config configs/experiments/retrieval_chunks.yaml

In [ ]:
# 10) Latency on CUDA: bf16, 4-bit NF4 and the API model
!MHRAG_LOAD_IN_4BIT=0 python -m scripts.bench_latency --config v2_hf_cuda
!python -m scripts.bench_latency --config v2_hf_cuda_4bit
!python -m scripts.bench_latency --config v2_api || echo "no GROQ_API_KEY"

### 10b) Local-model generation (the laptop experiment, moved to Kaggle)
`configs/experiments/generation_local.yaml`: the deployed CPU model **Qwen2.5-1.5B-Instruct (GGUF Q4_K_M, llama.cpp)**
plus the original v1 models **GPT-2** and **BART-large-CNN**, each run without retrieval, with v1-style naive RAG and
with the full pipeline, on FAQ-Gen, the out-of-scope set and 50 Counsel-Gen questions.

llama.cpp needs `llama-cpp-python`. The next cell tries a prebuilt CUDA wheel first. If there isn't one for this CUDA
version, it compiles with CUDA, which takes about 10–20 minutes.

In [ ]:
# Install llama.cpp bindings with CUDA offload (prebuilt wheel if available, otherwise build from source)
import subprocess, sys
def have_llama():
    return subprocess.run([sys.executable, "-c", "import llama_cpp"], capture_output=True).returncode == 0
if not have_llama():
    !pip -q install llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
if not have_llama():
    !CMAKE_ARGS="-DGGML_CUDA=on" pip -q install llama-cpp-python --no-binary llama-cpp-python
print("llama_cpp available:", have_llama())

In [ ]:
# Run the local-model generation experiment (checkpointed; re-run to resume)
import os
os.environ["MHRAG_LLAMACPP_GPU_LAYERS"] = "-1"   # offload all layers to the GPU
!MHRAG_LOAD_IN_4BIT=0 python -m scripts.run_eval --config configs/experiments/generation_local.yaml

### 11) Generation benchmark (the long step)
About 20 models × 3 settings × about 250 questions. On a T4, 7–9B models in 4-bit take roughly 1–2 hours each, so plan on
2–4 sessions. Answers are checkpointed per (model, setting, dataset). Re-running the cell after a restart continues where it
stopped (with the previous output attached, see cell 4).

For a session-sized chunk, edit `models:` in `configs/experiments/generation_gpu.yaml`, or run one model at a time with
`--models` (see the next cell). Use `--limit 20` for a quick smoke run.

In [ ]:
# Run the whole list...
!python -m scripts.run_eval --config configs/experiments/generation_gpu.yaml
# ...or a subset per session, e.g.:
# !python -m scripts.run_eval --config configs/experiments/generation_gpu.yaml --models qwen2.5-7b-instruct llama-3.1-8b-instruct

In [ ]:
# 12) Human-evaluation sheets (blinded, randomised) from the generation results
!python -m eval.human_eval.make_sheets --exp gpu --raters 3 --n 60

In [ ]:
# 13) Paper numbers + package everything into the notebook output
!python -m scripts.paper_numbers
import shutil, os
for sub in ("paper/tables", "paper/numbers.tex", "eval/human_eval/out"):
    src = os.path.join("/kaggle/working/mhrag", sub)
    dst = os.path.join(RESULTS, "_repo_" + sub.replace("/", "_"))
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    elif os.path.exists(src):
        shutil.copy(src, dst)
shutil.make_archive("/kaggle/working/mhrag_results", "zip", RESULTS)
print("Download /kaggle/working/mhrag_results.zip from the Output tab (or commit the notebook).")
print("Locally: unzip into the repo's results/ and copy _repo_paper_tables/* to paper/tables/.")